In [2]:
import xarray as xr
import sys
sys.path.append("../src")
from load_data import load_scenario_transmission_capacity
#trans_cap = load_scenario_transmission_capacity("../data/raw/scenario_data/barrier_policy_alpine_pv,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc")
#print(trans_cap)

In [ ]:
import xarray as xr
from load_data import get_ch_foreign_transmission_techs

nc_path = "../data/raw/scenario_data/barrier_policy_alpine_pv,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc"
ds = xr.open_dataset(nc_path)

foreign_techs = get_ch_foreign_transmission_techs(ds)
print(foreign_techs)

In [3]:
prod_coords = ds.coords["loc_tech_carriers_prod"].values
con_coords = ds.coords["loc_tech_carriers_con"].values

matched_prod = [c for c in prod_coords if any(c.startswith(t) for t in foreign_techs)]
matched_con = [c for c in con_coords if any(c.startswith(t) for t in foreign_techs)]

print(len(matched_prod), len(matched_con))
print(matched_prod[:10])
print(matched_con[:10])

NameError: name 'ds' is not defined

In [ ]:
import numpy as np

pairs_to_check = [
    "GE::ac_transmission:FRA::electricity",
    "FRA::ac_transmission:GE::electricity",
]

for p in pairs_to_check:
    prod_sum = ds["carrier_prod"].sel(loc_tech_carriers_prod=p).sum(dim="timesteps").values
    con_sum = ds["carrier_con"].sel(loc_tech_carriers_con=p).sum(dim="timesteps").values
    print(p)
    print("  carrier_prod sum:", prod_sum)
    print("  carrier_con sum:", con_sum)

GE::ac_transmission:FRA::electricity
  carrier_prod sum: 5305.674897962059
  carrier_con sum: -2207.4787239628185
FRA::ac_transmission:GE::electricity
  carrier_prod sum: 2177.3197872889264
  carrier_con sum: -5379.165946081884


In [ ]:
import pandas as pd

In [ ]:
def get_ch_import_carrier_techs(ds) -> list[str]:
    """loc_tech_carriers_prod Einträge, bei denen CH-Kanton Ursprung und Ausland Ziel ist
    -> carrier_prod hierauf = Import in die Schweiz."""
    prod_coords = ds.coords["loc_tech_carriers_prod"].values
    result = []
    for c in prod_coords:
        origin, _, rest = c.partition("::")
        if "ac_transmission" not in rest:
            continue
        dest = rest.partition(":")[2].partition("::")[0]
        if origin not in FOREIGN_CODES and dest in FOREIGN_CODES:
            result.append(c)
    return result


def load_scenario_import(nc_path) -> pd.DataFrame:
    ds = xr.open_dataset(nc_path)
    import_techs = get_ch_import_carrier_techs(ds)
    imports = (
        ds["carrier_prod"]
        .sel(loc_tech_carriers_prod=import_techs)
        .sum(dim="timesteps")
        .to_pandas()
        .rename("import_mwh")
        .reset_index()
    )
    ds.close()

    imports[["canton", "tech_carrier"]] = imports["loc_tech_carriers_prod"].str.split("::", n=1, expand=True)
    imports["language_region"] = imports["canton"].map(CANTON_LANGUAGE)
    return imports

from config import CANTON_LANGUAGE, FOREIGN_CODES

imports = load_scenario_import(nc_path)
print(imports)

In [ ]:
national_import_mwh = imports["import_mwh"].sum()
print(national_import_mwh)

39397.2564661289


In [ ]:
print(ds.coords["timesteps"].values[0])
print(ds.coords["timesteps"].values[-1])
print(len(ds.coords["timesteps"]))

2016-01-01T00:00:00.000000000
2016-12-31T20:00:00.000000000
2196


In [ ]:
import os
import xarray as xr

scenario_dir = "../data/raw/scenario_data/"
files = sorted(os.listdir(scenario_dir))
print(files)

['barrier_baseline,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_baseline_no_self_sufficiency_result_4H.nc', 'barrier_policy_alpine_pv,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_chp,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_h2_imports_only,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_mix,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_remix,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_roof_pv,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_sccgt_daccs,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_sccgt_no_h2,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_sccgt_plus_h2,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc', 'barrier_policy_wind_x2,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc']


In [ ]:
all_loc_techs = ds.coords["loc_techs"].values
fuel_candidates = [t for t in all_loc_techs if any(kw in t.lower() for kw in ["gas", "supply", "import", "biomass", "hydrogen", "h2", "oil", "coal"])]
print(len(fuel_candidates))
for t in sorted(set(fuel_candidates)):
    print(t)

618
AG::buildings_biofuel_boiler
AG::buildings_biofuel_boiler_heat_storage_small
AG::buildings_biofuel_boiler_tech_heat_to_demand
AG::buildings_methane_boiler
AG::buildings_methane_boiler_heat_storage_small
AG::buildings_methane_boiler_tech_heat_to_demand
AG::electrode_boiler
AG::electrode_boiler_heat_storage_big
AG::electrode_boiler_tech_heat_to_demand
AG::hydrogen_electricity_storage
AG::hydrogen_storage
AG::hydrogen_to_liquids
AG::hydrogen_to_methane
AG::import_syndiesel
AG::import_synmethane
AG::industry_biofuel_boiler
AG::industry_biofuel_boiler_heat_storage_big
AG::industry_biofuel_boiler_tech_heat_to_demand
AG::industry_methane_boiler
AG::industry_methane_boiler_heat_storage_big
AG::industry_methane_boiler_tech_heat_to_demand
AG::supply_biofuel
AG::supply_waste
AG::waste_boiler
AG::waste_boiler_heat_storage_big
AG::waste_boiler_tech_heat_to_demand
AI_AR::buildings_biofuel_boiler
AI_AR::buildings_biofuel_boiler_heat_storage_small
AI_AR::buildings_biofuel_boiler_tech_heat_to_deman

In [ ]:
prod_coords = ds.coords["loc_tech_carriers_prod"].values
fuel_import_coords = [c for c in prod_coords if "import_syndiesel" in c or "import_synmethane" in c]
print(len(fuel_import_coords))
print(sorted(fuel_import_coords)[:10])

50
[np.str_('AG::import_syndiesel::diesel'), np.str_('AG::import_synmethane::methane'), np.str_('AI_AR::import_syndiesel::diesel'), np.str_('AI_AR::import_synmethane::methane'), np.str_('AUT::import_synmethane::methane'), np.str_('BE::import_syndiesel::diesel'), np.str_('BE::import_synmethane::methane'), np.str_('BL_BS::import_syndiesel::diesel'), np.str_('BL_BS::import_synmethane::methane'), np.str_('DEU::import_synmethane::methane')]


In [ ]:
from load_data import load_scenario_fuel_import, aggregate_national


In [3]:
import sys
sys.path.append("../src")
from load_data import load_scenario_fuel_import, aggregate_national
nc_path = "../data/raw/scenario_data/barrier_policy_alpine_pv,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc"

fuel = load_scenario_fuel_import(nc_path)
print(len(fuel))
print(fuel.head(10))
print(aggregate_national(fuel, "fuel_import_mwh"))

46
              loc_tech_carriers_prod  fuel_import_mwh canton  \
0    BL_BS::import_syndiesel::diesel         0.000262  BL_BS   
1  BL_BS::import_synmethane::methane         0.000412  BL_BS   
2     FR::import_synmethane::methane         0.000412     FR   
3       TI::import_syndiesel::diesel         0.000262     TI   
4     LU::import_synmethane::methane         0.000412     LU   
5     UR::import_synmethane::methane         0.000412     UR   
6       FR::import_syndiesel::diesel         0.000262     FR   
7     SG::import_synmethane::methane         0.000412     SG   
8     SH::import_synmethane::methane         0.000412     SH   
9     TI::import_synmethane::methane         0.000412     TI   

                 tech_carrier language_region  
0    import_syndiesel::diesel           DE-CH  
1  import_synmethane::methane           DE-CH  
2  import_synmethane::methane           FR-CH  
3    import_syndiesel::diesel             NaN  
4  import_synmethane::methane           DE-CH  
5  i

In [4]:
import sys
sys.path.append("../src")
import xarray as xr

nc_path_h2 = "../data/raw/scenario_data/barrier_policy_h2_imports_only,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc"
ds_h2 = xr.open_dataset(nc_path_h2)

all_loc_techs_h2 = ds_h2.coords["loc_techs"].values
fuel_candidates_h2 = sorted(set(t for t in all_loc_techs_h2 if any(kw in t.lower() for kw in ["hydrogen", "methane", "diesel", "gas_import", "fossil"])))
print(len(fuel_candidates_h2))
for t in fuel_candidates_h2:
    print(t)
ds_h2.close()

642
AG::biofuel_to_diesel
AG::biofuel_to_methane
AG::buildings_methane_boiler
AG::buildings_methane_boiler_heat_storage_small
AG::buildings_methane_boiler_tech_heat_to_demand
AG::chp_methane
AG::diesel_transmission:BE
AG::diesel_transmission:BL_BS
AG::diesel_transmission:LU
AG::diesel_transmission:SO
AG::diesel_transmission:ZG
AG::diesel_transmission:ZH
AG::hydrogen_electricity_storage
AG::hydrogen_storage
AG::hydrogen_to_liquids
AG::hydrogen_to_methane
AG::import_hydrogen
AG::import_syndiesel
AG::import_synmethane
AG::industry_methane_boiler
AG::industry_methane_boiler_heat_storage_big
AG::industry_methane_boiler_tech_heat_to_demand
AG::methane_transmission:BE
AG::methane_transmission:BL_BS
AG::methane_transmission:LU
AG::methane_transmission:SO
AG::methane_transmission:ZG
AG::methane_transmission:ZH
AG::storage_diesel
AG::storage_methane
AI_AR::biofuel_to_diesel
AI_AR::biofuel_to_methane
AI_AR::buildings_methane_boiler
AI_AR::buildings_methane_boiler_heat_storage_small
AI_AR::buildin

In [5]:
fuel_h2 = load_scenario_fuel_import(nc_path_h2)
print(len(fuel_h2))
print(aggregate_national(fuel_h2, "fuel_import_mwh"))

69
18709.53323987263


In [1]:
import sys
sys.path.append("../src")
from load_data import load_scenario_capacity, compute_technology_mix_shares

nc_path = "../data/raw/scenario_data/barrier_policy_alpine_pv,neighbours_fixed_by_EP2050_baseline_2016_result_4H.nc"
capacity_df = load_scenario_capacity(nc_path)
shares = compute_technology_mix_shares(capacity_df)
print(shares)

ImportError: cannot import name 'compute_technology_mix_shares' from 'load_data' (/Users/josua/Documents/ETH/Semester Project/swiss-winter-deficit-choice-mapping-jm/notebooks/../src/load_data.py)